# 📚 GUÍA COMPLETA: ORDEN CORRECTO DE ANÁLISIS DE DATOS

## 🎯 FLUJO GENERAL DESDE CERO HASTA MODELADO

## 📋 SECUENCIA COMPLETA PASO A PASO

```
1. LIMPIEZA DE DATOS (antes del split)
   - Eliminar duplicados
   - Convertir tipos de datos
   - Codificar variables categóricas
   - Reemplazar valores imposibles (ej: 0 en glucosa)
   ↓
   
2. TRAIN/TEST SPLIT ⭐ (MOMENTO CRÍTICO)
   - Dividir antes de cualquier transformación estadística
   - Evitar data leakage
   ↓
   
3. IMPUTACIÓN (solo fit en train)
   - Entrenar imputer solo con X_train
   - Aplicar transform a X_test
   ↓
   
4. CORRELACIONES (solo en train) 🔴 PRIMERO
   - Calcular matriz de correlación
   - Eliminar variables con |r| > 0.8
   ↓
   
5. VIF (solo en train)
   - Calcular Variance Inflation Factor
   - Eliminar variables con VIF > 10
   ↓
   
6. MÉTODOS DE SELECCIÓN (solo en train) 🔵 DESPUÉS
   - Random Forest Feature Importance
   - Permutation Importance
   - SHAP values
   ↓
   
7. ESCALADO (fit en train, transform en test)
   - StandardScaler o RobustScaler
   ↓
   
8. MODELADO
   - Entrenar modelo final
```

---

# 🔴 PASO 4: CORRELACIONES (PRIMERO)

## ¿Por qué primero?
- ✅ Detecta **multicolinealidad** entre variables independientes (X)
- ✅ Elimina **redundancia** antes de entrenar modelos
- ✅ Más **eficiente computacionalmente** (no requiere entrenar modelos)
- ✅ Evita **problemas** en modelos lineales sensibles a correlación
- ✅ Reduce **dimensionalidad** desde el inicio

## 📊 CÓDIGO: Análisis de Correlaciones

In [ ]:
# 1. CALCULAR MATRIZ DE CORRELACIÓN (solo con X_train_imputado)
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

corr_matrix = X_train_imputado.corr()

# 2. VISUALIZAR CON HEATMAP
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, linewidths=0.5, cbar_kws={"shrink": .8})
plt.title('Matriz de Correlación - Variables Independientes', fontsize=16, pad=15)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# 3. IDENTIFICAR PARES DE VARIABLES ALTAMENTE CORRELACIONADAS (|r| > 0.8)
umbral_correlacion = 0.8

# Crear lista para almacenar pares correlacionados
pares_correlacionados = []

for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        correlacion = corr_matrix.iloc[i, j]
        if abs(correlacion) > umbral_correlacion:
            pares_correlacionados.append({
                'Variable_1': corr_matrix.columns[i],
                'Variable_2': corr_matrix.columns[j],
                'Correlación': round(correlacion, 3)
            })

# Mostrar resultados
if len(pares_correlacionados) > 0:
    df_pares = pd.DataFrame(pares_correlacionados)
    print("="*70)
    print(f"⚠️ VARIABLES ALTAMENTE CORRELACIONADAS (|r| > {umbral_correlacion})")
    print("="*70)
    print(df_pares)
    print("\n💡 RECOMENDACIÓN: Eliminar una de las variables de cada par")
else:
    print("="*70)
    print(f"✅ No hay variables con correlación > {umbral_correlacion}")
    print("="*70)

In [ ]:
# 4. ELIMINAR VARIABLES CORRELACIONADAS
# Ejemplo: Si 'emp.var.rate' y 'euribor3m' están correlacionadas
# Decidir cuál eliminar basándote en interpretación o importancia previa

variables_a_eliminar = ['emp.var.rate', 'euribor3m']  # EJEMPLO - ajustar según tus datos

# Eliminar en train
X_train_limpio = X_train_imputado.drop(variables_a_eliminar, axis=1, errors='ignore')

# Eliminar en test
X_test_limpio = X_test_imputado.drop(variables_a_eliminar, axis=1, errors='ignore')

print(f"✅ Variables eliminadas: {variables_a_eliminar}")
print(f"📊 Forma original: {X_train_imputado.shape}")
print(f"📊 Forma nueva: {X_train_limpio.shape}")

---

# 🟡 PASO 5: VIF (Variance Inflation Factor)

## ¿Qué es VIF?
- Mide cuánto se "infla" la varianza de un coeficiente debido a la multicolinealidad
- **VIF = 1**: No hay correlación
- **VIF = 1-5**: Correlación moderada (aceptable)
- **VIF = 5-10**: Correlación alta (precaución)
- **VIF > 10**: Multicolinealidad severa (eliminar variable)

## 📊 CÓDIGO: Cálculo de VIF

In [ ]:
from statsmodels.tools.tools import add_constant
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Agregar constante (requerido para VIF)
X_vif = add_constant(X_train_limpio)

# Calcular VIF para cada variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) 
                   for i in range(X_vif.shape[1])]

# Eliminar la constante de los resultados
vif_data = vif_data[vif_data["Variable"] != "const"]

# Ordenar por VIF descendente
vif_data = vif_data.sort_values("VIF", ascending=False)

print("="*60)
print("📊 VARIANCE INFLATION FACTOR (VIF)")
print("="*60)
print(vif_data)
print("\n" + "="*60)

# Identificar variables problemáticas
variables_altas = vif_data[vif_data["VIF"] > 10]
if len(variables_altas) > 0:
    print(f"⚠️ VARIABLES CON VIF > 10 (Eliminar):")
    print(variables_altas)
else:
    print("✅ No hay variables con VIF > 10")

---

# 🔵 PASO 6: MÉTODOS DE SELECCIÓN DE VARIABLES (DESPUÉS)

## ¿Por qué después?
- ✅ Trabaja con variables **ya "limpias"** sin correlación
- ✅ Resultados **más confiables** (no sesgados por multicolinealidad)
- ✅ **Más rápido** (menos variables para analizar)
- ✅ Evita que el modelo "elija arbitrariamente" entre variables correlacionadas

## Métodos a usar:
1. **Random Forest Feature Importance**
2. **Permutation Importance**
3. **SHAP Values**

## 📊 MÉTODO 1: Random Forest Feature Importance

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Entrenar Random Forest SOLO con datos de train (sin correlaciones)
model_rf = RandomForestClassifier(random_state=42, n_jobs=-1).fit(X_train_limpio, Y_train)

# Obtener importancias
importances = model_rf.feature_importances_
importances_pct = (importances / importances.sum()) * 100

# Crear DataFrame
df_rf_imp = pd.DataFrame({
    'feature': X_train_limpio.columns,
    'rf_importance': importances_pct
}).sort_values(by='rf_importance', ascending=False)

# Calcular importancia acumulada
df_rf_imp['rf_importance_acum'] = df_rf_imp['rf_importance'].cumsum()

print("="*70)
print("🌲 RANDOM FOREST - FEATURE IMPORTANCE")
print("="*70)
print(df_rf_imp)
print("\n" + "="*70)

## 📊 MÉTODO 2: Permutation Importance

In [ ]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.inspection import permutation_importance

# Crear conjunto de validación
X_train1, X_val, y_train1, y_val = train_test_split(
    X_train_limpio, Y_train, test_size=0.2, random_state=42
)

# Entrenar XGBoost
model_xgb = XGBClassifier(objective='binary:logistic', random_state=42).fit(X_train1, y_train1)

# Calcular Permutation Importance
perm = permutation_importance(model_xgb, X_val, y_val, 
                              n_repeats=10, random_state=42, 
                              n_jobs=-1, scoring='roc_auc')

# Crear DataFrame
df_perm_imp = pd.DataFrame({
    'feature': X_train_limpio.columns,
    'perm_imp': perm.importances_mean * 100
}).sort_values('perm_imp', ascending=False)

print("="*70)
print("🔀 PERMUTATION IMPORTANCE")
print("="*70)
print(df_perm_imp)
print("\n" + "="*70)

## 📊 MÉTODO 3: SHAP Values

In [ ]:
import lightgbm as lgb
import shap

# Entrenar LightGBM
model_lgbm = lgb.LGBMClassifier(random_state=42, n_jobs=-1).fit(X_train1, y_train1)

# Calcular SHAP values
explainer = shap.Explainer(model_lgbm, X_val)
shap_vals = explainer(X_val).values

# Importancia promedio absoluta
imp_shap = np.abs(shap_vals).mean(axis=0)
imp_shap_pct = (imp_shap / imp_shap.sum()) * 100

# Crear DataFrame
df_shap_imp = pd.DataFrame({
    "feature": X_val.columns,
    "shap_imp": imp_shap_pct
}).sort_values('shap_imp', ascending=False)

print("="*70)
print("🎯 SHAP VALUES - FEATURE IMPORTANCE")
print("="*70)
print(df_shap_imp)
print("\n" + "="*70)

# Gráfico SHAP
shap.summary_plot(shap_vals, X_val, plot_type="bar")

## 🔗 COMBINACIÓN DE LOS 3 MÉTODOS

In [ ]:
# Combinar los 3 métodos en un solo DataFrame
df_importances = (
    df_rf_imp
    .merge(df_perm_imp, on='feature', how='outer')
    .merge(df_shap_imp, on='feature', how='outer')
).sort_values('rf_importance', ascending=False)

# Calcular promedio de importancia
df_importances['importancia_promedio'] = df_importances[
    ['rf_importance', 'perm_imp', 'shap_imp']
].mean(axis=1)

# Ordenar por promedio
df_importances_sorted = df_importances.sort_values('importancia_promedio', ascending=False)

print("="*90)
print("📊 COMBINACIÓN DE LOS 3 MÉTODOS DE SELECCIÓN")
print("="*90)
print(df_importances_sorted.round(2))
print("\n" + "="*90)

# Filtrar variables con importancia promedio >= 2%
umbral_importancia = 2.0
df_filtradas = df_importances_sorted[df_importances_sorted['importancia_promedio'] >= umbral_importancia]

print(f"\n✅ VARIABLES SELECCIONADAS (Importancia promedio >= {umbral_importancia}%):")
print("="*90)
print(df_filtradas[['feature', 'importancia_promedio']].round(2))
print(f"\n📊 Total de variables seleccionadas: {len(df_filtradas)} de {len(df_importances_sorted)}")

---

# ⚠️ RESUMEN: ¿QUÉ PASA SI LO HACES AL REVÉS?

## ❌ PROBLEMA: Selección → Correlaciones

### Consecuencias:
1. **Resultados inconsistentes**: El modelo puede elegir arbitrariamente entre variables correlacionadas
2. **Mayor tiempo de cómputo**: Entrenas con variables redundantes
3. **Interpretación difícil**: Dos variables dicen lo mismo
4. **Sesgo en importancia**: Variables correlacionadas compiten entre sí

### Ejemplo:
```
Si Glucose y Insulin_log están correlacionadas (r = 0.85):

❌ MAL (Selección primero):
  - Random Forest elige Glucose como importante (85%)
  - En otra ejecución elige Insulin_log (82%)
  - Resultados inestables

✅ BIEN (Correlaciones primero):
  - Eliminas Insulin_log
  - Random Forest solo ve Glucose
  - Resultados estables y confiables
```

---

# 📋 CHECKLIST FINAL

## ✅ Orden Correcto:

| Paso | Acción | Herramienta | ¿Cuándo? |
|------|--------|------------|----------|
| 1 | Limpieza básica | Pandas | Antes del split |
| 2 | **Train/Test Split** | `train_test_split` | **AQUÍ** |
| 3 | Imputación | MissForest/SimpleImputer | Solo fit en train |
| 4 | **Correlaciones** | `corr()`, heatmap | **PRIMERO** (solo train) |
| 5 | VIF | `variance_inflation_factor` | Después correlaciones |
| 6 | **Selección Variables** | RF, Permutation, SHAP | **DESPUÉS** (solo train) |
| 7 | Escalado | StandardScaler | Fit en train, transform en test |
| 8 | Modelado | LogisticRegression, etc. | Con variables seleccionadas |

---

## 🎯 REGLA DE ORO:

**TODO lo que "aprende" de los datos debe hacerse SOLO con train después del split**

- ✅ **Correlaciones**: Calculadas con `X_train_imputado`
- ✅ **VIF**: Calculado con `X_train_limpio`
- ✅ **Feature Importance**: Modelos entrenados con `X_train_limpio`
- ✅ **Escalado**: `scaler.fit()` solo con `X_train`

---

## 💡 TIP FINAL:

En tu proyecto de diabetes o clasificación bancaria:

1. Divide primero con `train_test_split`
2. Imputa valores faltantes (fit solo en train)
3. Analiza correlaciones y elimina variables redundantes
4. Calcula VIF para confirmar
5. Aplica RF + Permutation + SHAP para seleccionar las mejores
6. Escala y modela 🚀

---

# 🔄 TRANSFORMACIONES: ¿ANTES O DESPUÉS DEL SPLIT?

## ✅ TRANSFORMACIONES ACEPTABLES ANTES DEL SPLIT

### Regla: Transformaciones que NO aprenden de los datos

Estas son **determinísticas** y se pueden aplicar antes del split sin riesgo de data leakage:

In [ ]:
# ✅ EJEMPLOS DE TRANSFORMACIONES ANTES DEL SPLIT:

# 1. TRANSFORMACIONES MATEMÁTICAS DETERMINÍSTICAS
df['Insulin_log'] = np.log1p(df['Insulin'])  # Logaritmo
df['BMI_sqrt'] = np.sqrt(df['BMI'])  # Raíz cuadrada
df['Age_squared'] = df['Age'] ** 2  # Potencia
df['DiabetesPedigreeFunction_sqrt'] = np.sqrt(df['DiabetesPedigreeFunction'])

# 2. CODIFICACIONES PREDEFINIDAS (MANUAL)
# Mapping ordinal manual
df['month'] = df['month'].map({
    'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 
    'may': 5, 'jun': 6, 'jul': 7, 'aug': 8,
    'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12
})

df['education'] = df['education'].map({
    'illiterate': 1,
    'basic.4y': 2,
    'basic.6y': 2,
    'basic.9y': 2,
    'high.school': 3,
    'professional.course': 4,
    'university.degree': 4
})

# 3. REEMPLAZAR VALORES IMPOSIBLES POR NaN
df['BloodPressure'] = df['BloodPressure'].replace(0, np.nan)
df['Glucose'] = df['Glucose'].replace(0, np.nan)
df['SkinThickness'] = df['SkinThickness'].replace(0, np.nan)

# 4. CREAR VARIABLES DERIVADAS
df['BMI_category'] = pd.cut(df['BMI'], bins=[0, 18.5, 25, 30, 100], 
                             labels=[1, 2, 3, 4])

print("✅ Transformaciones determinísticas aplicadas antes del split")
print("✅ No hay data leakage porque no se aprende de los datos")

### ¿Por qué es seguro?

- ✅ **No hay data leakage** porque no usas estadísticas de los datos (media, mediana, moda, etc.)
- ✅ La transformación es **igual para todos los datos** (no depende del conjunto)
- ✅ Es **reproducible** y **determinística** (siempre da el mismo resultado)
- ✅ No contamina el conjunto de test con información del conjunto de train

---

## ❌ TRANSFORMACIONES QUE DEBEN IR DESPUÉS DEL SPLIT

### Regla: Transformaciones que APRENDEN estadísticas de los datos

Estas deben hacerse solo con train y luego aplicarse a test:

In [ ]:
# ❌ EJEMPLOS DE TRANSFORMACIONES DESPUÉS DEL SPLIT:

# 1. IMPUTACIÓN CON ESTADÍSTICAS
from sklearn.impute import SimpleImputer

# ❌ MAL - Data leakage
imputer = SimpleImputer(strategy='median')
df_imputed = imputer.fit_transform(df)  # ← Aprende mediana de TODO el dataset
X_train, X_test = train_test_split(df_imputed)  # ← Test contaminado

# ✅ BIEN - Sin data leakage
X_train, X_test = train_test_split(df)
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)  # ← Aprende solo de train
X_test_imp = imputer.transform(X_test)  # ← Aplica lo aprendido de train


# 2. ESCALADO (STANDARDSCALER, MINMAXSCALER, ROBUSTSCALER)
from sklearn.preprocessing import StandardScaler

# ❌ MAL
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df)  # ← Aprende media/std de TODO
X_train, X_test = train_test_split(df_scaled)

# ✅ BIEN
X_train, X_test = train_test_split(df)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)  # ← Aprende de train
X_test_sc = scaler.transform(X_test)  # ← Usa parámetros de train


# 3. TARGET ENCODING
# ❌ MAL - Aprende tasas de conversión del dataset completo
target_encoding = df.groupby('job')['y'].mean()
df['job_encoded'] = df['job'].map(target_encoding)
X_train, X_test = train_test_split(df)

# ✅ BIEN - Aprende solo de train
X_train, X_test = train_test_split(df)
target_encoding = X_train.groupby('job')['y'].mean()
X_train['job_encoded'] = X_train['job'].map(target_encoding)
X_test['job_encoded'] = X_test['job'].map(target_encoding)


print("❌ Estas transformaciones causan DATA LEAKAGE si se hacen antes del split")
print("✅ Siempre hacer fit() solo con train, luego transform() en test")

---

## ⚠️ CASO ESPECIAL: Reemplazar 0 por Mediana

### Situación: Valores imposibles (ej: Glucosa = 0)

**Técnicamente mejor después del split, pero impacto mínimo si se hace antes:**

In [ ]:
# ⚠️ OPCIÓN 1: ANTES DEL SPLIT (Impacto mínimo pero técnicamente no ideal)
mediana_bp = df['BloodPressure'].median()  # ← Usa info de train Y test
df['BloodPressure'] = df['BloodPressure'].replace(0, mediana_bp)
X_train, X_test = train_test_split(df)

print(f"⚠️ Mediana calculada con todo el dataset: {mediana_bp}")
print("Impacto: Mínimo, pero técnicamente hay mini data leakage")


# ✅ OPCIÓN 2: DESPUÉS DEL SPLIT (Ideal, sin data leakage)
X_train, X_test = train_test_split(df)

# Calcular mediana solo con train
mediana_bp_train = X_train['BloodPressure'][X_train['BloodPressure'] > 0].median()

# Aplicar a ambos conjuntos
X_train['BloodPressure'] = X_train['BloodPressure'].replace(0, mediana_bp_train)
X_test['BloodPressure'] = X_test['BloodPressure'].replace(0, mediana_bp_train)

print(f"\n✅ Mediana calculada solo con train: {mediana_bp_train}")
print("Sin data leakage - Test no contamina train")


# ALTERNATIVA: Reemplazar por NaN antes del split (mejor práctica)
df['BloodPressure'] = df['BloodPressure'].replace(0, np.nan)
df['Glucose'] = df['Glucose'].replace(0, np.nan)
# Luego dividir y usar imputer (que aprenderá solo de train)

---

## 📊 TABLA RESUMEN

| Transformación | ¿Antes del split? | Razón |
|----------------|-------------------|-------|
| **`np.log1p()`** | ✅ **Sí** | No aprende de los datos - Determinística |
| **`np.sqrt()`** | ✅ **Sí** | No aprende de los datos - Determinística |
| **`x ** 2`** | ✅ **Sí** | No aprende de los datos - Determinística |
| **Mapeo manual** (1,2,3) | ✅ **Sí** | No aprende de los datos - Predefinido |
| **One-Hot Encoding** | ✅ **Sí** | Si las categorías son conocidas |
| **Reemplazar por NaN** | ✅ **Sí** | No aprende estadísticas |
| **Target Encoding** | ⚠️ **Después** | Aprende tasas del target (data leakage) |
| **Reemplazar 0 por mediana** | ⚠️ **Después (ideal)** | Aprende estadística del dataset |
| **StandardScaler** | ❌ **Después** | Aprende media y desviación estándar |
| **MinMaxScaler** | ❌ **Después** | Aprende mínimo y máximo |
| **RobustScaler** | ❌ **Después** | Aprende cuartiles |
| **SimpleImputer** | ❌ **Después** | Aprende mediana/media/moda |
| **MissForest** | ❌ **Después** | Aprende patrones de imputación |
| **KNNImputer** | ❌ **Después** | Aprende vecinos cercanos |

---

## 🎯 REGLA DE ORO SIMPLIFICADA

### ¿Usa `.fit()` o calcula estadísticas (media, mediana, moda, etc.)?

- **NO** → ✅ Puedes hacerlo **ANTES** del split
- **SÍ** → ❌ Debes hacerlo **DESPUÉS** del split (solo fit con train)

---

## 💡 EJEMPLO PRÁCTICO: Proyecto de Diabetes

```python
# ✅ ANTES DEL SPLIT
df['Insulin_log'] = np.log1p(df['Insulin'])  # Transformación matemática
df['BMI_sqrt'] = np.sqrt(df['BMI'])  # Transformación matemática
df['BloodPressure'] = df['BloodPressure'].replace(0, np.nan)  # Marcar como faltante

# ⭐ DIVIDIR
X = df.drop('Outcome', axis=1)
Y = df['Outcome']
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# ❌ DESPUÉS DEL SPLIT
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)  # fit solo con train
X_test_imp = imputer.transform(X_test)  # transform con parámetros de train

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_imp)  # fit solo con train
X_test_sc = scaler.transform(X_test_imp)  # transform con parámetros de train
```

In [ ]:
# OPCIÓN 1: Crear variable de ratio 
X_train_imputado['BMI_SkinThickness_ratio'] = X_train_imputado['BMI'] / (X_train_imputado['SkinThickness'] + 1)
X_test_imputado['BMI_SkinThickness_ratio'] = X_test_imputado['BMI'] / (X_test_imputado['SkinThickness'] + 1)

In [ ]:
# Comprobacion con el método Logit de statsmodels
var_test=X_train_imputado[['BMI', 'SkinThickness']]
var_test['BMI_SkinThickness_ratio']=var_test['BMI']/(var_test['SkinThickness']+1)
# 1. Instanciar el modelo (solo define la función, aún no entrena)
modelo_logit = sm.Logit(Y_train, sm.add_constant(var_test))

# 2. Ajustar el modelo (entrenamiento)
resultados = modelo_logit.fit()
# Muestra la tabla completa de resultados
print(resultados.summary())

In [ ]:
# OPCIÓN 2: Crear variable de producto
X_train_imputado['BMI_SkinThickness_producto'] = X_train_imputado['BMI'] * X_train_imputado['SkinThickness']
X_test_imputado['BMI_SkinThickness_producto'] = X_test_imputado['BMI'] * X_test_imputado['SkinThickness']

In [ ]:
# Comprobacion con el método Logit de statsmodels
var_test=X_train_imputado[['BMI', 'SkinThickness']]
var_test['BMI_SkinThickness_producto']=var_test['BMI']*var_test['SkinThickness']
# 1. Instanciar el modelo (solo define la función, aún no entrena)
modelo_logit = sm.Logit(Y_train, sm.add_constant(var_test))

# 2. Ajustar el modelo (entrenamiento)
resultados = modelo_logit.fit()
# Muestra la tabla completa de resultados
print(resultados.summary())

# SELECCIÓN DE HIPERPARÁMETROS PARA RANDOM FOREST: JUSTIFICACIÓN CIENTÍFICA

## 📊 n_estimators (Número de Árboles): [5, 10, 25, 50, 100, 200]

### Criterio de selección:

**1. Valores bajos (5-10):**
- Permite ver el comportamiento con **pocos árboles**
- Útil para datasets pequeños o cuando buscas **rapidez**
- Mayor variabilidad entre ejecuciones
- Rápido para prototipos y pruebas

**2. Valores medios (25-50):**
- **Rango típico recomendado** para empezar
- Balance entre rendimiento y tiempo de entrenamiento
- Suficiente diversidad para estabilizar predicciones
- Punto dulce para la mayoría de problemas

**3. Valores altos (100-200):**
- **Estándar en producción** (sklearn usa 100 por defecto)
- Mayor estabilidad, menor varianza
- Rendimiento marginal después de cierto punto (plateau)
- Convergencia garantizada de las predicciones

### ¿Por qué NO más de 200?
- Después de 100-200 árboles, la mejora es **mínima** (ley de rendimientos decrecientes)
- Mayor costo computacional sin beneficio proporcional
- En dataset de 768 muestras, 200 árboles ya es **más que suficiente**
- Cada árbol adicional aporta información redundante

---

### 📈 Comportamiento esperado según n_estimators:

| n_estimators | Varianza | Tiempo | Overfitting | Uso típico |
|--------------|----------|--------|-------------|------------|
| 5-10 | Alta ⚠️ | Rápido ⚡ | Bajo | Exploración inicial |
| 25-50 | Media | Moderado | Bajo-Medio | Producción ligera |
| 100-200 | Baja ✓ | Lento | Medio | Producción estándar |
| 500+ | Muy baja | Muy lento 🐌 | Alto | Competencias/casos extremos |

## 🌳 max_depth (Profundidad Máxima): [2, 5, 10, 15, 20, None]

### Criterio de selección:

**1. Depth = 2 (Muy bajo - Stumps):**
- **Árboles débiles** o "decision stumps"
- Previene overfitting agresivamente
- Útil cuando tienes pocas features (7 en nuestro caso)
- Similar al modelo simple que ya probamos
- Alta interpretabilidad

**2. Depth = 5-10 (Moderado - RECOMENDADO):**
- **Rango óptimo** para la mayoría de problemas
- Captura interacciones complejas sin memorizar
- Balance óptimo bias-varianza
- Suficiente profundidad para patrones no lineales
- Mantiene interpretabilidad razonable

**3. Depth = 15-20 (Alto):**
- Permite capturar **patrones muy complejos**
- Riesgo de overfitting si no hay suficientes datos
- Necesario para datasets con muchas features
- Para relaciones altamente no lineales
- Sacrifica interpretabilidad

**4. Depth = None (Sin límite - Control experimental):**
- Árboles crecen hasta **hojas puras** (o hasta min_samples_leaf)
- **Máximo overfitting potencial**
- Útil para comparar: "¿Qué pasa si dejo que memorice todo?"
- Sirve como baseline de peor caso
- Muestra límite superior de complejidad

### ¿Por qué estos límites específicos?
- Con **7 features**, profundidad mayor a 20 es redundante
- Matemáticamente: profundidad 20 → hasta 2²⁰ = ~1 millón de hojas posibles
- Pero solo tenemos 768 muestras → árboles profundos memorizarían
- `None` actúa como **control experimental** para ver efecto de restricciones

---

### 🔍 Interpretación de profundidad:

| max_depth | Splits máximos | Hojas máximas | Complejidad | Riesgo overfitting |
|-----------|----------------|---------------|-------------|--------------------|
| 2 | 2 | 4 | Muy baja | Muy bajo ✓ |
| 5 | 5 | 32 | Baja | Bajo ✓ |
| 10 | 10 | 1,024 | Media | Medio ⚠️ |
| 15 | 15 | 32,768 | Alta | Alto ⚠️ |
| 20 | 20 | 1,048,576 | Muy alta | Muy alto 🔴 |
| None | Ilimitado | Ilimitado | Extrema | Extremo 🔴 |

**Nota:** Con `min_samples_leaf=20`, las hojas reales serán mucho menores que el máximo teórico.

## 🎯 Justificación específica para el dataset de diabetes

| Característica del Dataset | Valor | Impacto en hiperparámetros |
|----------------------------|-------|----------------------------|
| **Tamaño total** | 768 muestras | n_estimators moderados suficientes (50-100) |
| **Train set** | ~614 muestras | Evita árboles muy profundos (≤10) |
| **Número de features** | 7 variables | max_depth bajo-medio óptimo (2-10) |
| **Balance de clases** | 65% No / 35% Sí | No requiere bosques gigantes |
| **Métrica prioritaria** | Recall (médico) | Profundidad media previene falsos negativos |
| **Tipo de problema** | Clasificación binaria | Menos complejo que multiclase |
| **Naturaleza de datos** | Mediciones médicas | Relaciones no lineales moderadas |

### 📐 Cálculos teóricos para nuestro caso:

**Regla de Breiman (creador de Random Forest):**
```
Profundidad óptima teórica = √(n_features) a log₂(n_features)

Para nuestro dataset:
- √7 ≈ 2.6
- log₂(7) ≈ 2.8

→ Rango teórico óptimo: depth 2-5
```

**Regla de sklearn (basada en experiencia):**
```
n_estimators default = 100
max_depth default = None (pero ajustar para prevenir overfitting)

Para datasets pequeños (<1000 muestras):
- n_estimators: 50-200
- max_depth: 5-15
```

**Regla empírica de muestras por hoja:**
```
Muestras por hoja mínimas = 2^max_depth × min_samples_leaf

Con min_samples_leaf = 20:
- depth=2  → 4 hojas   → 80 muestras mínimas ✓
- depth=5  → 32 hojas  → 640 muestras (límite para 614) ⚠️
- depth=10 → 1024 hojas → 20,480 muestras (imposible con 614) ✗

→ Profundidad práctica máxima ≈ 5-7 para nuestro dataset
```

## 📚 Fundamentos científicos y literatura académica

### 1. **Breiman, L. (2001) - Random Forests (Machine Learning, 45(1), 5-32)**
- Artículo original que introduce Random Forest
- **Recomendaciones:**
  - "100-200 árboles son suficientes para la mayoría de problemas"
  - "Más allá de 200, la mejora es marginal"
  - "Profundidad óptima: relacionada con √(n_features)"
- **Citas clave:**
  > "The forest error rate depends on two things: the correlation between any two trees in the forest, and the strength of each individual tree in the forest. Reducing the correlation increases the forest accuracy."

### 2. **Hastie, Tibshirani & Friedman - The Elements of Statistical Learning**
- Libro de referencia en Machine Learning
- **Sobre n_estimators:**
  - Error rate se estabiliza típicamente entre 50-200 árboles
  - Más árboles no causan overfitting (a diferencia de boosting)
  - Trade-off es computacional, no estadístico
- **Sobre max_depth:**
  - Árboles individuales deben ser "suficientemente complejos"
  - Pero no tan profundos que memoricen (high variance)
  - Bootstrap + feature sampling ya reduce overfitting

### 3. **Sklearn Documentation (Pedregosa et al., 2011)**
- Implementación estándar de Random Forest
- **Defaults:**
  ```python
  n_estimators = 100  # Basado en experimentos empíricos
  max_depth = None     # Pero agregar restricciones en práctica
  min_samples_split = 2
  min_samples_leaf = 1
  ```
- **Recomendaciones para tuning:**
  - Empezar con defaults
  - Ajustar max_depth si hay overfitting
  - Aumentar n_estimators si hay alta varianza
  - Usar cross-validation para validar

### 4. **Oshiro et al. (2012) - How Many Trees in a Random Forest?**
- Estudio específico sobre número óptimo de árboles
- **Hallazgos:**
  - 64-128 árboles óptimo para 29 datasets UCI
  - Después de 200, mejora < 0.1% en accuracy
  - Tiempo crece linealmente, beneficio no
- **Conclusión:** "128 árboles es un buen default para la mayoría"

### 5. **Probst et al. (2019) - Hyperparameters and Tuning Strategies for Random Forest**
- Meta-análisis de 38 datasets
- **Rankings de importancia de hiperparámetros:**
  1. `mtry` (max_features) - MÁS IMPORTANTE
  2. `min.node.size` (min_samples_leaf)
  3. `max_depth` 
  4. `n_estimators` - MENOS IMPORTANTE (si > 50)
- **Implicación:** n_estimators > 50-100 es suficiente, foco en profundidad

## 🔬 Método científico aplicado: Exploración → Refinamiento

### FASE 1: Exploración amplia (lo que implementamos)
```python
n_estimators = [5, 10, 25, 50, 100, 200]  # 6 valores espaciados logarítmicamente
max_depth = [2, 5, 10, 15, 20, None]       # 6 valores que cubren todo el espectro

# Total: 36 combinaciones
# Estrategia: Grid Search exhaustivo en rango amplio
```

**Ventajas:**
- ✅ Cubre todo el espacio de hiperparámetros
- ✅ No asume conocimiento previo del óptimo
- ✅ Detecta patrones inesperados
- ✅ Visualizaciones revelan tendencias claras

**Desventajas:**
- ⚠️ Computacionalmente costoso (36 modelos)
- ⚠️ Puede incluir combinaciones absurdas

---

### FASE 2: Refinamiento (si necesitamos más precisión)

**Escenario A: Si óptimo está en n=50, depth=5**
```python
n_estimators = [40, 45, 50, 55, 60]    # ±20% alrededor del óptimo
max_depth = [4, 5, 6, 7]                # ±1-2 niveles
# Total: 20 combinaciones más enfocadas
```

**Escenario B: Si necesitamos balance exploración-explotación**
```python
# Random Search: probar 20 combinaciones aleatorias
from sklearn.model_selection import RandomizedSearchCV

param_distributions = {
    'n_estimators': [10, 25, 50, 75, 100, 150, 200],
    'max_depth': [2, 3, 5, 7, 10, 15, None],
    'min_samples_leaf': [10, 20, 30, 50],
    'max_features': [3, 5, 7, 'sqrt', 'log2']
}
# Prueba 20 combinaciones aleatorias en lugar de 7×7×4×5 = 980
```

**Escenario C: Bayesian Optimization (más avanzado)**
```python
# Usa resultados previos para guiar búsqueda
# Converge más rápido que Grid o Random
# Librerías: Optuna, Hyperopt, scikit-optimize
```

---

### 📊 Espaciamiento logarítmico vs lineal

**¿Por qué [5, 10, 25, 50, 100, 200] y no [5, 45, 85, 125, 165, 200]?**

```
Espaciamiento LOGARÍTMICO (lo que usamos):
5 → 10 (×2)
10 → 25 (×2.5)
25 → 50 (×2)
50 → 100 (×2)
100 → 200 (×2)

Ventaja: Cubre rangos de magnitud diferentes
- Detecta si 5 vs 10 árboles importa (puede ser crucial)
- También prueba si 100 vs 200 importa (generalmente no)

Espaciamiento LINEAL:
5 → 45 (+40)
45 → 85 (+40)
...
Desventaja: Desperdicia recursos en rangos altos donde diferencia es mínima
```

**Analogía:** Es como ajustar volumen de música
- Pasar de 5% a 10% = gran cambio perceptible
- Pasar de 95% a 100% = apenas se nota
- Escala logarítmica captura mejor la percepción humana

## 🏭 Práctica en industria y competencias (Kaggle)

### Configuraciones comunes por contexto:

| Contexto | n_estimators | max_depth | min_samples_leaf | Justificación |
|----------|--------------|-----------|------------------|---------------|
| **Prototipo rápido** | 10-25 | 5 | 50 | Balance velocidad-calidad |
| **Producción estándar** | 100 | 10 | 20 | Default robusto |
| **Modelo médico** | 50-100 | 3-7 | 30-50 | Interpretabilidad + recall |
| **Finanzas (fraud)** | 200-500 | 8-12 | 10 | Alta precisión requerida |
| **Competencia Kaggle** | 500-1000 | None | 1 | Máximo rendimiento, no importa tiempo |
| **Dispositivos móviles** | 10-20 | 3-5 | 100 | Modelo ligero |

### 📈 Casos de estudio reales:

**1. Sistema de diagnóstico de cáncer (Wisconsin Breast Cancer)**
```python
# Dataset similar: 569 muestras, 30 features
# Configuración ganadora:
n_estimators = 100
max_depth = 5
min_samples_leaf = 5
# Accuracy: 97.2%
# Justificación: Médico requiere interpretabilidad
```

**2. Predicción de diabetes (Pima Indians - nuestro dataset)**
```python
# Estudios publicados usan:
n_estimators = 50-100  # Suficiente para 768 muestras
max_depth = 5-10       # Balance complejidad-generalización
# Recall típico: 70-75% (nuestra métrica objetivo)
```

**3. Sistema de recomendación Netflix (millones de muestras)**
```python
n_estimators = 500+    # Dataset masivo justifica más árboles
max_depth = 15-20      # Patrones complejos requieren profundidad
# Trade-off: Precisión vs latencia aceptable
```

### 🎯 Reglas de oro de la industria:

1. **Regla del 1% (Ley de rendimientos decrecientes)**
   - Si aumentar hiperparámetros mejora < 1% → no vale la pena
   - Ejemplo: 100 árboles = 75% accuracy, 200 árboles = 75.3% → quedarse con 100

2. **Regla del 10x (Tiempo de entrenamiento)**
   - Si duplicar recursos aumenta tiempo > 10x → buscar alternativas
   - Random Forest escala linealmente con n_estimators ✓
   - Profundidad excesiva causa escalado exponencial ✗

3. **Regla del dataset size**
   ```
   n_estimators_sugerido ≈ log₁₀(n_muestras) × 20
   
   Para nuestro caso:
   768 muestras → log₁₀(768) ≈ 2.9 → 2.9 × 20 ≈ 58 árboles
   ```

4. **Regla de features**
   ```
   max_depth_sugerido ≈ 2 × log₂(n_features)
   
   Para nuestro caso:
   7 features → log₂(7) ≈ 2.8 → 2 × 2.8 ≈ 5-6 niveles
   ```

## 🧪 Validación experimental: ¿Qué esperamos encontrar?

### Hipótesis basadas en teoría:

**H1: Efecto de n_estimators**
```
Hipótesis: Test accuracy aumenta rápido hasta ~50-100 árboles, luego plateau
Predicción visual: Curva logarítmica que se aplana

  Accuracy
    |     /-------- (plateau después de 100)
    |    /
    |   /
    |  /
    | /
    |/________________
        50  100  200  n_estimators
```

**H2: Efecto de max_depth**
```
Hipótesis: Train accuracy sube con depth, test accuracy forma U invertida
Predicción: Óptimo en depth 5-10 para nuestro dataset

  Accuracy
    |        ___
    |       /   \    ← Test (U invertida)
    |      /     \___
    |     /
    |____/____________ ← Train (siempre sube)
       2  5  10  20  max_depth
       
  Underfitting   Óptimo   Overfitting
```

**H3: Interacción n_estimators × max_depth**
```
Hipótesis: Más árboles compensan parcialmente overfitting de profundidad
Predicción en heatmap:
- Esquina (n=5, d=2):   Underfit → accuracy baja
- Esquina (n=200, d=20): Overfit → accuracy media
- Centro (n=50-100, d=5-10): ÓPTIMO → accuracy alta
```

---

### 📋 Checklist de validación post-experimento:

Después de ejecutar el grid search, verificar:

- [ ] **¿Se cumple el plateau de n_estimators?**
  - Si accuracy sigue subiendo después de 200 → considerar más árboles
  - Si plateau antes de 50 → dataset pequeño, reducir puede ser suficiente

- [ ] **¿Dónde está el óptimo de max_depth?**
  - Si óptimo en depth=2 → problema simple, mantener bajo
  - Si óptimo en depth=20 o None → problema complejo, pero cuidado overfitting

- [ ] **¿Diferencia train-test crece con complejidad?**
  - Debe aumentar con max_depth ✓
  - No debe aumentar mucho con n_estimators ✓

- [ ] **¿Recall se comporta diferente que accuracy?**
  - En problemas médicos, puede requerir profundidad diferente
  - Verificar trade-off accuracy vs recall

- [ ] **¿Hay configuraciones anómalas?**
  - Si (n=5, d=20) > (n=200, d=5) → algo está mal en datos/código
  - Resultados deben ser consistentes con teoría

## 💡 Alternativas y cuándo ajustar los rangos

### Escenario 1: Dataset más grande (>10,000 muestras)
```python
n_estimators = [100, 200, 500, 1000]  # Más datos justifican más árboles
max_depth = [10, 15, 20, 25, None]    # Mayor profundidad sin overfitting
```

### Escenario 2: Dataset más pequeño (<500 muestras)
```python
n_estimators = [5, 10, 25, 50]     # Menos árboles suficientes
max_depth = [2, 3, 5, 7]            # Restricción fuerte contra overfitting
min_samples_leaf = [50, 100]        # Hojas más grandes
```

### Escenario 3: Muchas features (>50 variables)
```python
n_estimators = [100, 200, 500]      # Más diversidad necesaria
max_depth = [15, 20, 25, None]      # Mayor profundidad para interacciones
max_features = ['sqrt', 'log2', 0.3]  # Feature sampling crítico
```

### Escenario 4: Recursos computacionales limitados
```python
n_estimators = [10, 25, 50]         # Menos modelos
max_depth = [3, 5, 10]              # Rango reducido
# Usar RandomizedSearchCV en lugar de GridSearchCV
```

### Escenario 5: Producción en tiempo real
```python
n_estimators = [5, 10, 20]          # Latencia crítica
max_depth = [2, 3, 5]               # Inferencia rápida
# Considerar árboles más simples o modelo distilling
```

---

## 🎓 Conclusión: ¿Por qué estos parámetros NO son arbitrarios?

### Resumen de fundamentos:

1. **Teoría matemática (Breiman, 2001)**
   - Convergencia estadística después de ~100-200 árboles
   - Profundidad óptima relacionada con √n_features

2. **Validación empírica (Oshiro et al., 2012)**
   - 64-128 árboles óptimo en 29 datasets UCI
   - Mejora marginal después de 200

3. **Análisis del dataset (nuestro caso específico)**
   - 768 muestras → n_estimators moderados
   - 7 features → max_depth bajo-medio
   - Problema médico → interpretabilidad importante

4. **Experiencia práctica (sklearn defaults + industria)**
   - n_estimators=100 es default probado en millones de casos
   - max_depth ajustable según overfitting observado

5. **Principio de parsimonia (Navaja de Occam)**
   - Modelo más simple que logra resultado aceptable
   - Balance complejidad-rendimiento-interpretabilidad

---

### 🔑 Lección clave:

> **Los rangos de hiperparámetros deben estar fundamentados en:**
> 1. Teoría estadística del algoritmo
> 2. Características del dataset específico
> 3. Literatura científica y casos previos
> 4. Restricciones prácticas (tiempo, recursos)
> 5. Objetivo del negocio (precisión vs interpretabilidad vs velocidad)

**No son números "al azar" → Son decisiones informadas basadas en 20+ años de investigación en Random Forests.**